In [2]:
import numpy as np
import pandas as pd

# read in file

In [2]:
parent_folder = '/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/'
trail = 'MokuLockInAmplifierData_20260629_143748.csv'

# extract info and data sections from original CSV

In [3]:
def extract_info_and_data(parent_path, block, trial_name):
    import pandas as pd
    original_data_path = parent_path+block+ "/" +trial_name

    # extract experiment info
    og_df = pd.read_csv(f"{original_data_path}.csv", skiprows=1)
    info_df = og_df[:8] # clip to only "info" section
    info_df = info_df.T # transpose so columns are titles, rows are data
    info_df.columns = info_df.iloc[0] # create new header row
    info_df = info_df[1:] # remove extra header row
    # save info df
    info_df.to_csv(parent_path+trial_name+"_info.csv", index = False)

    # create and save data df
    data_df = pd.read_csv(f"{original_data_path}.csv", header = 9) # remove info rows, keep only data rows with Time, and probes as headers

    # rename Probe A and B to quadrature and pre-mixed wave
    data_df.columns = ['Time (s)', 'Quadrature (V)', "Pre-Mixed Wave (V)"]

    # save new data df
    data_df.to_csv(parent_path+trial_name+"_data.csv", index = False)

    # print new csv file names

    filepaths = [parent_path+trial_name+"_info.csv", parent_path+trial_name+"_data.csv"]

    # print("experiment INFO saved to: \n", parent_path+trial_name+"_info.csv \n")
    # print("experiment DATA saved to: \n", parent_path+trial_name+"_data.csv")

    return filepaths


In [4]:
parent_folder = '/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/'
trial = 'MokuLockInAmplifierData_20260629_143748'
data_and_info_paths = extract_info_and_data(parent_folder, "", trial)
info = pd.read_csv(data_and_info_paths[0])
data = pd.read_csv(data_and_info_paths[1])
data.head()

,Time (s),Quadrature (V),Pre-Mixed Wave (V)
0,0.000,0.173009,-0.004100
1,0.001,0.173009,-0.004100
2,0.002,0.173009,-0.004100
3,0.003,0.173009,-0.004133
4,0.004,0.173040,-0.004133


# reduce data df to every tenth row

In [10]:
# take every tenth row of data entry
dec_index = list(range(0, len(data), 10))
data = data.iloc[dec_index, :]
data.head()

,Time (s),Probe A (V),Probe B (V)
0,0.00,0.173009,-0.004100
10,0.01,0.173009,-0.004133
20,0.02,0.173009,-0.004133
30,0.03,0.173009,-0.004133
40,0.04,0.173009,-0.004100


# putting together multiple trial runs

In [ ]:
parent_folder = '/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/'
trial = 'MokuLockInAmplifierData_20260629_143748'
data_and_info_paths = extract_info_and_data(parent_folder, trial)
info = pd.read_csv(data_and_info_paths[0])
data = pd.read_csv(data_and_info_paths[1])
info

In [16]:
import os
location = "/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/"
block = "a1"

def combine_trials(loc, extension, variable, block):
    # extract trials for desired variable from data folder
    import os
    folder_path = loc+block+"/"
    print(folder_path)
    files = [
        filename for filename in os.listdir(folder_path)
        if filename.endswith(extension) and variable in filename
    ]
    print('made files list')
    files = [os.path.splitext(f)[0] for f in files]
    print('took off extension')

    # extract data contents from files
    trial_data_paths = []
    for f in files:
        trial_n_contents_path = extract_info_and_data(parent_folder, block, f)
        trial_data_paths.append(trial_n_contents_path[1])

    # empty data frames for creating combined df, averages at each time of the three trials
    combined_df = pd.DataFrame()

    trial = pd.read_csv(trial_data_paths[0])
    combined_df["Time_s"] = trial["Time (s)"]
    col_names_q = []
    col_names_pw = []
    
    for t in list(range(0, len(trial_data_paths))):
        trial_df = pd.read_csv(trial_data_paths[t])
        quad_col = f"Quad_V_Trial{t+1}"
        col_names_q.append(quad_col)
        combined_df[quad_col] = trial_df['Quadrature (V)']
        premix_col = f"PreMix_V_Trial{t+1}"
        col_names_pw.append(premix_col)
        combined_df[premix_col] = trial_df['Pre-Mixed Wave (V)']

    combined_df["Avg. Quadrature (V)"] = combined_df[col_names_q].mean(axis=1)
    combined_df["Avg. Pre-Mixed Wave (V)"] = combined_df[col_names_pw].mean(axis=1)
    column_names = list(combined_df.columns)
    new_order = [column_names[0]]+column_names[-2:] + column_names[1:-2]
    combined_df = combined_df[new_order]

    combined_df.to_csv(folder_path+f"{block}combinedtrials.csv")
    print("New combined trials dataframe saved to", folder_path+f"{block}combinedtrials.csv")

    return combined_df

combo = combine_trials(location, ".csv", "x", "a1")
combo

/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/
made files list
took off extension
New combined trials dataframe saved to /Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/combinedtrials.csv


,Time_s,Avg. Quadrature (V),Avg. Pre-Mixed Wave (V),Quad_V_Trial1,PreMix_V_Trial1,Quad_V_Trial2,PreMix_V_Trial2,Quad_V_Trial3,PreMix_V_Trial3
0,0.000,0.000000,0.000267,0.0,0.000000,0.000000,0.000267,0.000000,0.000533
1,0.001,-0.000011,-0.000178,0.0,0.000000,0.000000,-0.000267,-0.000033,-0.000267
2,0.002,0.000000,-0.000089,0.0,-0.000267,0.000000,0.000000,0.000000,0.000000
3,0.003,-0.000011,-0.000178,0.0,-0.000267,0.000000,0.000000,-0.000033,-0.000267
4,0.004,0.000000,-0.000089,0.0,0.000000,0.000000,0.000000,0.000000,-0.000267
...,...,...,...,...,...,...,...,...,...
29995,29.995,0.000000,0.000000,0.0,-0.000267,0.000000,0.000267,0.000000,0.000000
29996,29.996,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
29997,29.997,0.000000,-0.000178,0.0,0.000000,0.000000,-0.000267,0.000000,-0.000267
29998,29.998,-0.000011,0.000178,0.0,0.000533,0.000000,0.000000,-0.000033,0.000000


,"quad_trial(0, [1, 2, 3])","quad_trial(1, [4, 5, 6])","quad_trial(2, [3, 4, 5])"
0,"[1, 2, 3]","[1, 2, 3]","[1, 2, 3]"
1,"[4, 5, 6]","[4, 5, 6]","[4, 5, 6]"
2,"[3, 4, 5]","[3, 4, 5]","[3, 4, 5]"
